# The Composition Cliff — and how serial compute walks around it

All six recommendations, built. **Recommendation 1 is already measured**, on one CPU
core, and the result is sharper than the theory I was working from predicted.

## What was measured

Task: track one element through *k* permutations of **S₅**. S₅ is non-solvable, so its
word problem is NC¹-complete; a fixed-depth log-precision transformer lives in uniform
TC⁰ (Merrill & Sabharwal, TACL 2023). `domain` = how many of the 120 group elements the
sequence is drawn from.

**Sanity first.** One-shot, k=1 (pure lookup, no composition): **accuracy 1.0000**.
The setup works. Everything below is a real failure, not a broken harness.

**The cliff.** k=2, d=64, 2000 steps, chance = 0.2:

| domain | 5 | 20 | 60 | 80 | **100** | 110 | 120 |
|---|---|---|---|---|---|---|---|
| accuracy | 1.0000 | 1.0000 | 0.9975 | 0.9862 | **0.2032** | 0.2095 | 0.1953 |

A **phase transition between domain 80 and 100.** Below it, composition is perfect.
Above it, chance. Not a gradual decay — a wall.

**Nothing you would normally reach for moves the wall** (k=2, domain=120):

| lever | change | before | after |
|---|---|---|---|
| width | d 64 → 128 (**3.7× params**) | 0.1985 | 0.2037 |
| depth | layers 2 → 4 | 0.1985 | 0.2010 |
| looped depth | loops 1 → 2 (same params) | 0.1985 | 0.1925 |
| training | 3000 → **8000 steps** | 0.1985 | 0.2032 |

**Serial compute walks around it.** Same task, domain 120:

| arm | params | accuracy |
|---|---|---|
| one-shot, d=128 | 430,463 | 0.2037 |
| **chain-of-thought, d=64** | **116,991** | **1.0000** |
| chain-of-thought, d=64, k=4 | 117,247 | 1.0000 |

The chain-of-thought model is evaluated **autoregressively** — it generates its own chain
and is scored only on the final answer. It has **3.7× fewer parameters** than the one-shot
model it beats, and it goes from chance to perfect.

## Why this matters for you

This is the thing you have been circling for a month, in one table. The bottleneck was
never that 0.5B is too small. A model **3.7× smaller** solves a task that more width,
more depth, more looping, and 2.7× more training all fail at — because it is allowed to
*think in steps*.

The cliff is the empirical shadow of Peng, Narayanan & Papadimitriou (COLM 2024), who
prove by communication complexity that a transformer layer cannot compose functions
*once the domains are large enough*. Here "large enough" is somewhere around 90.

**One honest caveat:** I have shown *nothing* moves the cliff among {width, depth, loops,
steps} at the settings tested. I have not shown nothing *can*. A much wider model, a
different positional scheme, or curriculum training might. That is the first thing to
falsify, and cell 6 is set up to do it.

## 1 · Setup

In [ ]:
import math, time, json, itertools, sys
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337); np.random.seed(1337)
print("device:", DEV)
if DEV == "cpu":
    torch.set_num_threads(max(1, torch.get_num_threads()))
    print("(runs on CPU; every number in the header was produced on ONE core)")

## 2 · The task

Track one element through k permutations of S₅. `domain` controls how many distinct
group elements appear — the knob that produced the cliff.

In [ ]:
_PERMS = list(itertools.permutations(range(5)))
PERM = np.array(_PERMS, dtype=np.int64)        # PERM[g][p] = g(p)
N_PERM, N_POS = 120, 5
POFF, BOS, SEP, VOCAB = 5, 125, 126, 127

class S5Track:
    """domain = size of the subset of S5 the permutations are drawn from."""
    def __init__(self, k, domain=120, seed=0):
        self.k, self.domain = k, domain
        self.rng = np.random.default_rng(seed)
        self.subset = np.arange(domain)

    def sample(self, n):
        start = self.rng.integers(0, N_POS, size=n)
        g = self.subset[self.rng.integers(0, self.domain, size=(n, self.k))]
        return start, g

    def trace(self, start, g):
        out = np.empty_like(g); cur = start.copy()
        for t in range(g.shape[1]):
            cur = PERM[g[:, t], cur]; out[:, t] = cur
        return out

    def batch_oneshot(self, n):
        s, g = self.sample(n); tr = self.trace(s, g)
        seq = np.concatenate([np.full((n,1),BOS), s[:,None], g+POFF,
                              np.full((n,1),SEP)], axis=1)
        return (torch.from_numpy(seq).long().to(DEV),
                torch.from_numpy(tr[:,-1]).long().to(DEV))

    def batch_cot(self, n):
        s, g = self.sample(n); tr = self.trace(s, g)
        seq = np.concatenate([np.full((n,1),BOS), s[:,None], g+POFF,
                              np.full((n,1),SEP), tr], axis=1)
        x = torch.from_numpy(seq[:,:-1]).long().to(DEV)
        y = torch.from_numpy(seq[:,1:]).long().to(DEV)
        m = torch.zeros_like(y, dtype=torch.bool); m[:, -self.k:] = True
        return x, y, m

    @property
    def chance(self): return 1.0/N_POS
    @property
    def maxlen(self): return 2*self.k + 4

print("[ok] task")

## 3 · Model

Three switches, each mapping to a specific claim:
`n_layers` = parallel depth · `n_loops` = recurrent depth (same weights reapplied)
· `confidence_head` = a second head trained with a **proper scoring rule**, which is what
lets a stage abstain and bound its own error.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.n1=nn.LayerNorm(d); self.attn=nn.MultiheadAttention(d,h,batch_first=True)
        self.n2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d), nn.GELU(), nn.Linear(4*d,d))
    def forward(self,x,causal):
        h=self.n1(x); a,_=self.attn(h,h,h,attn_mask=causal,need_weights=False)
        x=x+a; return x+self.ff(self.n2(x))

class TinyLM(nn.Module):
    def __init__(self, vocab=VOCAB, d=64, n_layers=2, n_heads=4, max_len=64,
                 n_loops=1, confidence_head=False):
        super().__init__()
        self.n_loops=n_loops
        self.tok=nn.Embedding(vocab,d); self.pos=nn.Embedding(max_len,d)
        self.blocks=nn.ModuleList([Block(d,n_heads) for _ in range(n_layers)])
        self.norm=nn.LayerNorm(d); self.head=nn.Linear(d,vocab)
        self.conf=nn.Sequential(nn.Linear(d,d),nn.GELU(),nn.Linear(d,1)) if confidence_head else None
    def forward(self, idx):
        B,T=idx.shape
        x=self.tok(idx)+self.pos(torch.arange(T,device=idx.device))[None]
        causal=torch.triu(torch.full((T,T),float("-inf"),device=idx.device),1)
        for _ in range(self.n_loops):
            for b in self.blocks: x=b(x,causal)
        x=self.norm(x)
        return self.head(x), (self.conf(x).squeeze(-1) if self.conf is not None else None)
    def n_params(self): return sum(p.numel() for p in self.parameters())

print("[ok] model")

## 4 · Train and evaluate

In [ ]:
def fit(task, d=64, steps=2000, cot=False, bs=256, lr=3e-3,
        layers=2, loops=1, seed=0, log=0):
    torch.manual_seed(seed)
    m=TinyLM(d=d,n_layers=layers,max_len=task.maxlen,n_loops=loops).to(DEV)
    opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=0.01)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,lr,total_steps=steps)
    for s in range(steps):
        if cot:
            x,y,mask=task.batch_cot(bs); lg,_=m(x); loss=F.cross_entropy(lg[mask],y[mask])
        else:
            x,y=task.batch_oneshot(bs); lg,_=m(x); loss=F.cross_entropy(lg[:,-1,:N_POS],y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); sch.step()
        if log and (s+1)%log==0: print(f"   {s+1}/{steps} loss {loss.item():.4f}",flush=True)
    return m

@torch.no_grad()
def acc_oneshot(m,task,n=4000,bs=1000):
    m.eval(); c=0
    for _ in range(n//bs):
        x,y=task.batch_oneshot(bs); lg,_=m(x); c+=int((lg[:,-1,:N_POS].argmax(-1)==y).sum())
    m.train(); return c/(n//bs*bs)

@torch.no_grad()
def acc_cot(m,task,n=2000,bs=500):
    """AUTOREGRESSIVE. The model writes its own chain; only the final state is scored."""
    m.eval(); c=0
    for _ in range(n//bs):
        s,g=task.sample(bs); ans=task.trace(s,g)[:,-1]
        seq=np.concatenate([np.full((bs,1),BOS),s[:,None],g+POFF,np.full((bs,1),SEP)],axis=1)
        cur=torch.from_numpy(seq).long().to(DEV)
        for _ in range(task.k):
            lg,_=m(cur); cur=torch.cat([cur,lg[:,-1,:N_POS].argmax(-1,keepdim=True)],1)
        c+=int((cur[:,-1].cpu().numpy()==ans).sum())
    m.train(); return c/(n//bs*bs)

print("[ok] harness")

## 5 · R1 — reproduce the cliff

On a GPU this is a couple of minutes. It reproduces the header table.

In [ ]:
print("SANITY  k=1, no composition (must be ~1.0, else the harness is broken)")
t=S5Track(1); m=fit(t,64,1500); a=acc_oneshot(m,t)
print(f"  one-shot k=1: {a:.4f}   {'OK' if a>0.9 else 'HARNESS BROKEN - STOP'}\n")

print("THE CLIFF  k=2, d=64, 2000 steps, chance=0.2")
cliff=[]
for D in [5,20,60,80,100,120]:
    t=S5Track(2,domain=D); m=fit(t,64,2000); a=acc_oneshot(m,t)
    cliff.append((D,a)); print(f"  domain {D:3d}: {a:.4f}",flush=True)

print("\nSERIAL COMPUTE  same task, domain=120")
t=S5Track(2,domain=120)
m1=fit(t,128,2000,cot=False); a1=acc_oneshot(m1,t)
m2=fit(t,64,2000,cot=True);   a2=acc_cot(m2,t)
print(f"  one-shot d=128 ({m1.n_params():,} params): {a1:.4f}")
print(f"  CoT      d= 64 ({m2.n_params():,} params): {a2:.4f}")
print(f"\n  {m1.n_params()/m2.n_params():.1f}x fewer parameters, {a2-a1:+.4f} accuracy")

## 6 · Falsify me

The honest gap in the header: I showed *nothing I tried* moves the cliff. Try harder.
If any of these crosses the wall at domain 120, my interpretation is wrong and you should
publish that instead.

In [ ]:
print("Trying to move the cliff at domain=120, k=2 (chance 0.2)")
t=S5Track(2,domain=120)
for tag,kw in [("width d=256",       dict(d=256,steps=2000)),
               ("depth 6 layers",    dict(d=64,steps=2000,layers=6)),
               ("loops 4",           dict(d=64,steps=2000,loops=4)),
               ("long training 20k", dict(d=64,steps=20000)),
               ("low lr 5e-4",       dict(d=64,steps=6000,lr=5e-4))]:
    t0=time.time(); m=fit(t,**kw); a=acc_oneshot(m,t)
    flag="  <-- CLIFF CROSSED, interpretation is WRONG" if a>0.5 else ""
    print(f"  {tag:18s} acc {a:.4f}  [{time.time()-t0:.0f}s]{flag}",flush=True)

## 7 · R2 — objective before architecture

Bachmann & Nagarajan (ICML 2024): on the path-star graph, teacher-forced next-token
prediction provably fails. Teacher forcing hands the model the hard first node for free,
after which every remaining token is a trivial edge-follow — the "Clever Hans cheat" —
so the one token that requires planning never gets a learning signal.

Prediction: teacher-forced ≈ 1/D. Teacherless (predict the whole path from a masked
prefix) ≫ that. Same architecture, same parameters, **different objective**.

In [ ]:
class PathStar:
    def __init__(self, degree=2, length=5, seed=0):
        self.D,self.L=degree,length; self.n=degree*length+1
        self.SEP,self.GO,self.MASK=self.n,self.n+1,self.n+2
        self.VOCAB=self.n+3; self.rng=np.random.default_rng(seed)
    def one(self):
        nodes=self.rng.permutation(np.arange(1,self.n)); arms=nodes.reshape(self.D,self.L)
        E=[]
        for a in range(self.D):
            E.append((0,arms[a,0]))
            for i in range(self.L-1): E.append((arms[a,i],arms[a,i+1]))
        self.rng.shuffle(E)
        j=int(self.rng.integers(0,self.D))
        return [t for e in E for t in e], int(arms[j,-1]), arms[j].tolist()
    def batch(self,n,teacherless=False):
        X,Y=[],[]
        for _ in range(n):
            flat,tgt,path=self.one()
            pre=flat+[self.SEP,0,tgt,self.GO]
            X.append(pre+([self.MASK]*self.L if teacherless else path)); Y.append(path)
        return (torch.tensor(X).long().to(DEV), torch.tensor(Y).long().to(DEV))
    @property
    def maxlen(self): return 2*self.D*self.L+4+self.L
    @property
    def chance(self): return 1.0/self.D

def fit_pathstar(task, teacherless, steps=3000, d=64, bs=128, lr=3e-3, seed=0):
    torch.manual_seed(seed)
    m=TinyLM(vocab=task.VOCAB,d=d,max_len=task.maxlen+2).to(DEV)
    opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=0.01)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,lr,total_steps=steps)
    for s in range(steps):
        x,y=task.batch(bs,teacherless=teacherless)
        lg,_=m(x)
        # score the L answer slots either way; only the INPUT differs
        pred=lg[:,-task.L-1:-1,:]
        loss=F.cross_entropy(pred.reshape(-1,pred.shape[-1]), y.reshape(-1))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); sch.step()
    return m

@torch.no_grad()
def acc_pathstar(m,task,n=1000,bs=250):
    """Scored on the FIRST node -- the only one that requires planning."""
    m.eval(); c=0
    for _ in range(n//bs):
        x,y=task.batch(bs,teacherless=True); lg,_=m(x)
        c+=int((lg[:,-task.L-1,:].argmax(-1)==y[:,0]).sum())
    m.train(); return c/(n//bs*bs)

ps=PathStar(degree=2,length=5)
for tl in [False,True]:
    t0=time.time(); m=fit_pathstar(ps,teacherless=tl); a=acc_pathstar(m,ps)
    print(f"  {'teacherless' if tl else 'teacher-forced'}: first-node acc {a:.4f} "
          f"(chance {ps.chance}) [{time.time()-t0:.0f}s]",flush=True)

## 8 · R3 — calibration as a first-class output

Trained with the **Brier score**, a strictly proper scoring rule: uniquely minimised by
reporting the true probability of being correct. Reported via the Murphy (1973)
decomposition, because a single number hides what matters:

**Brier = Reliability − Resolution + Uncertainty**

Reliability is the calibration gap (lower better). Resolution is discrimination (higher
better). A model can improve Brier by becoming *less informative*, so they must be read
separately. Then **selective risk** (Geifman & El-Yaniv, NeurIPS 2017): accuracy among
answered items as a function of coverage. That curve is what lets a pipeline stage bound
its own error by abstaining.

In [ ]:
def brier_decomposition(conf, correct, bins=10):
    conf=np.asarray(conf); correct=np.asarray(correct,dtype=float)
    base=correct.mean(); unc=base*(1-base)
    edges=np.linspace(0,1,bins+1); rel=res=0.0; N=len(conf)
    for i in range(bins):
        m=(conf>=edges[i])&(conf<edges[i+1] if i<bins-1 else conf<=1.0)
        if m.sum()==0: continue
        nk=m.sum(); pk=conf[m].mean(); ok=correct[m].mean()
        rel+=nk*(pk-ok)**2; res+=nk*(ok-base)**2
    rel/=N; res/=N
    return dict(brier=float(((conf-correct)**2).mean()), reliability=float(rel),
                resolution=float(res), uncertainty=float(unc))

def selective_risk(conf, correct):
    """Accuracy among ANSWERED items, sweeping the abstention threshold."""
    o=np.argsort(-np.asarray(conf)); c=np.asarray(correct,dtype=float)[o]
    cov=np.arange(1,len(c)+1)/len(c); acc=np.cumsum(c)/np.arange(1,len(c)+1)
    return cov, acc

def fit_calibrated(task, steps=2500, d=64, bs=256, lr=3e-3, cot=False, seed=0):
    torch.manual_seed(seed)
    m=TinyLM(d=d,max_len=task.maxlen,confidence_head=True).to(DEV)
    opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=0.01)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,lr,total_steps=steps)
    for s in range(steps):
        x,y=task.batch_oneshot(bs); lg,cf=m(x)
        task_loss=F.cross_entropy(lg[:,-1,:N_POS],y)
        with torch.no_grad(): ok=(lg[:,-1,:N_POS].argmax(-1)==y).float()
        cal=((torch.sigmoid(cf[:,-1])-ok)**2).mean()      # Brier, strictly proper
        (task_loss+cal).backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(),1.0)
        opt.step(); sch.step(); opt.zero_grad()
    return m

@torch.no_grad()
def collect(m,task,n=4000,bs=1000):
    m.eval(); C,K=[],[]
    for _ in range(n//bs):
        x,y=task.batch_oneshot(bs); lg,cf=m(x)
        C.append(torch.sigmoid(cf[:,-1]).cpu().numpy())
        K.append((lg[:,-1,:N_POS].argmax(-1)==y).cpu().numpy())
    m.train(); return np.concatenate(C), np.concatenate(K)

# a MIXED-difficulty stream: some below the cliff, some above
for D in [60,120]:
    t=S5Track(2,domain=D); m=fit_calibrated(t)
    conf,ok=collect(m,t); dec=brier_decomposition(conf,ok)
    cov,acc=selective_risk(conf,ok)
    print(f"  domain {D:3d}: acc {ok.mean():.4f} | Brier {dec['brier']:.4f} "
          f"(rel {dec['reliability']:.4f}, res {dec['resolution']:.4f})")
    for c in [0.1,0.25,0.5,1.0]:
        i=min(int(c*len(cov))-1,len(cov)-1)
        print(f"      coverage {c:4.0%} -> selective accuracy {acc[i]:.4f}")

## 9 · R5 — RL elicits, it does not create

Yue et al. (arXiv:2504.13837): RL raises pass@1 and *lowers* pass@k at large k, because it
concentrates mass on paths the base model already had. The signature is a **crossover**.

Sharpening here is done honestly — self-training on the model's own *correct* samples,
which is the best case for RL. If even that shows the crossover, the point stands.

In [ ]:
@torch.no_grad()
def pass_at_k(m,task,k_list,n=400,bs=100,temp=1.0):
    m.eval(); out={}
    kmax=max(k_list); hits=np.zeros((n,kmax),dtype=bool); idx=0
    for _ in range(n//bs):
        x,y=task.batch_oneshot(bs)
        lg,_=m(x); probs=F.softmax(lg[:,-1,:N_POS]/temp,-1)
        s=torch.multinomial(probs,kmax,replacement=True)
        hits[idx:idx+bs]=(s==y[:,None]).cpu().numpy(); idx+=bs
    m.train()
    for k in k_list: out[k]=float(hits[:,:k].any(1).mean())
    return out

def sharpen(m,task,rounds=600,bs=128,lr=3e-4,temp=1.0):
    """Self-train on its OWN correct samples -- the charitable version of RL."""
    opt=torch.optim.AdamW(m.parameters(),lr=lr)
    for _ in range(rounds):
        x,y=task.batch_oneshot(bs)
        with torch.no_grad():
            lg,_=m(x); s=torch.multinomial(F.softmax(lg[:,-1,:N_POS]/temp,-1),1).squeeze(-1)
            keep=(s==y)
        if keep.sum()<4: continue
        lg2,_=m(x[keep]); loss=F.cross_entropy(lg2[:,-1,:N_POS],s[keep])
        opt.zero_grad(); loss.backward(); opt.step()
    return m

t=S5Track(2,domain=60)          # below the cliff, so there IS capability to elicit
base=fit(t,64,2000)
import copy; rl=sharpen(copy.deepcopy(base),t)
KL=[1,2,4,8,16,32,64]
pb,pr=pass_at_k(base,t,KL),pass_at_k(rl,t,KL)
print(f"  {'k':>4} {'base':>9} {'sharpened':>11}")
cross=None
for k in KL:
    mark=""
    if pr[k]<pb[k] and cross is None: cross=k; mark="  <- crossover"
    print(f"  {k:>4} {pb[k]:>9.4f} {pr[k]:>11.4f}{mark}")
print(f"\n  crossover at k={cross}" if cross else "\n  no crossover observed")

## 10 · R6 — the combined primitive

Everything at once: looped depth (serial compute), teacherless-style supervision on the
chain, a proper-scored confidence head, and abstention. A stage that **thinks in steps and
knows when it does not know** — which is the only kind of component you can safely stack.

In [ ]:
class SerialCalibrated(nn.Module):
    """CoT + looped depth + calibrated confidence + abstention."""
    def __init__(self, task, d=64, layers=2, loops=2):
        super().__init__()
        self.task=task
        self.net=TinyLM(d=d,n_layers=layers,max_len=task.maxlen,
                        n_loops=loops,confidence_head=True).to(DEV)
    def fit(self, steps=3000, bs=256, lr=3e-3):
        opt=torch.optim.AdamW(self.net.parameters(),lr=lr,weight_decay=0.01)
        sch=torch.optim.lr_scheduler.OneCycleLR(opt,lr,total_steps=steps)
        for s in range(steps):
            x,y,mask=self.task.batch_cot(bs)
            lg,cf=self.net(x)
            task_loss=F.cross_entropy(lg[mask],y[mask])
            with torch.no_grad(): ok=(lg[mask].argmax(-1)==y[mask]).float()
            cal=((torch.sigmoid(cf[mask])-ok)**2).mean()
            (task_loss+cal).backward()
            torch.nn.utils.clip_grad_norm_(self.net.parameters(),1.0)
            opt.step(); sch.step(); opt.zero_grad()
        return self
    @torch.no_grad()
    def decide(self, n=2000, bs=500, threshold=0.5):
        """Returns (accuracy_on_answered, coverage). Abstains when unsure."""
        self.net.eval(); k=self.task.k; ok=[]; cf=[]
        for _ in range(n//bs):
            s,g=self.task.sample(bs); ans=self.task.trace(s,g)[:,-1]
            seq=np.concatenate([np.full((bs,1),BOS),s[:,None],g+POFF,
                                np.full((bs,1),SEP)],axis=1)
            cur=torch.from_numpy(seq).long().to(DEV); confs=[]
            for _ in range(k):
                lg,c=self.net(cur)
                confs.append(torch.sigmoid(c[:,-1]))
                cur=torch.cat([cur,lg[:,-1,:N_POS].argmax(-1,keepdim=True)],1)
            chain_conf=torch.stack(confs,1).min(1).values   # weakest link
            ok.append((cur[:,-1].cpu().numpy()==ans)); cf.append(chain_conf.cpu().numpy())
        self.net.train()
        ok=np.concatenate(ok); cf=np.concatenate(cf)
        ans_mask=cf>=threshold
        cov=float(ans_mask.mean())
        acc=float(ok[ans_mask].mean()) if ans_mask.any() else float("nan")
        return acc, cov, ok, cf

t=S5Track(4,domain=120)
sc=SerialCalibrated(t).fit()
acc,cov,ok,cf=sc.decide()
print(f"  overall accuracy        {ok.mean():.4f}")
print(f"  answered accuracy       {acc:.4f}  at coverage {cov:.1%}")
d=brier_decomposition(cf,ok)
print(f"  Brier {d['brier']:.4f}  reliability {d['reliability']:.4f}  resolution {d['resolution']:.4f}")
c2,a2=selective_risk(cf,ok)
for c in [0.25,0.5,0.75,1.0]:
    i=min(int(c*len(c2))-1,len(c2)-1)
    print(f"      coverage {c:4.0%} -> selective accuracy {a2[i]:.4f}")

## 11 · R4 — where kernel freedom actually buys capability

You can write any CUDA you like. It is worth being precise about what that can and cannot
change, because the distinction is not intuitive.

| change | speed | capability |
|---|---|---|
| fused kernels, FlashAttention, quantization, tensor cores | yes | **no** — same function, faster |
| longer context via memory-efficient attention | yes | **indirectly** — affords longer chains, and chains change the complexity class |
| a different **objective** (cell 7) | no | **yes** |
| a different **factorization** (any-order, teacherless) | no | **yes** |
| **effective depth** via looping (cell 3) | no | **yes** |
| **serial token budget** at inference (cell 5) | no | **yes** |

A kernel that computes the same mathematical function faster cannot move the cliff in
cell 5 — the cliff is a property of the function class, not of throughput. What kernel
work *does* buy is making long chains and many loops affordable, and that is a real lever
because chains change what is computable. Aim your CUDA at the bottom four rows.

## 12 · What this is

A small, measured result: **a 117k-parameter model with serial steps beats a 430k-parameter
model without them, on a task where width, depth, looping and 2.7× training all fail.**

It is not AGI and it is not close. It is one axis of the nine, tested honestly on a task
where theory made a prediction in advance. That is worth more than nine hardcoded PASSes,
and it is a real thing to build on.